# TFM V3 — frozen MTP-encoder linear probe

This is the final predefined TFM transfer version. It reuses the exact V1 token cache, passes token maps through the official four-layer MTP-pretrained TFM encoder with every encoder parameter frozen, averages reader features per sentence, and fits only a balanced linear probe.

```text
V1 token maps -> frozen official MTP encoder -> reader feature
              -> mean across readers -> sentence feature
              -> nested-CV linear probe -> aligned vs shuffled gate
```

| Fixed choice | V3 value |
| --- | --- |
| Encoder | Official TFM 64x4, MTP-pretrained on TUAB/TUEV/CHB-MIT |
| Encoder updates | None |
| ZuCo montage adapter | Consecutive 16-channel groups; group features averaged by channel count |
| Classifier | Standardized, class-balanced L2 logistic regression |
| Evaluation | 5 unseen-sentence folds, seeds 42/52/62 |
| Control | Split-local shuffled sentence features plus majority |
| Decision | Same locked five-part gate used for V2 |


## Run instructions

Select **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. The first V3 run extracts frozen features and saves one atomic file per subject. A disconnect loses at most the subject currently being encoded; rerunning all cells reuses completed subjects. No raw EEG loading or tokenizer inference is repeated.


In [ ]:
# 1) Fetch this codebase and install only missing official-encoder dependencies.
from pathlib import Path
import importlib.metadata, importlib.util, os, subprocess, sys

PROJECT_URL = "https://github.com/parmisbathayan/EEGTokenizer.git"
PROJECT_ROOT = Path("/content/EEGTokenizer")

def run(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "--depth", "1", PROJECT_URL, str(PROJECT_ROOT)])
else:
    run(["git", "pull", "--ff-only"], cwd=PROJECT_ROOT)
requirements = {
    "einops": "einops==0.8.0",
    "linear_attention_transformer": "linear-attention-transformer==0.19.1",
    "timm": "timm==1.0.14",
}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    run([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Official-encoder dependencies already available")
project_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
os.chdir(PROJECT_ROOT / "tfm")
print("Working directory:", Path.cwd())
print("Project revision:", project_revision)


In [ ]:
# 2) Mount Drive and use the established Data / CachedArtifacts / Results layout.
from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis")
CACHE_ROOT = THESIS_ROOT / "CachedArtifacts/eeg_tokenizer/tfm"
RESULTS_ROOT = THESIS_ROOT / "Results/eeg_tokenizer/tfm"
TOKEN_CACHE = CACHE_ROOT / "tokens_v1"
PACKED_TOKEN_CACHE = CACHE_ROOT / "token_records_v2_packed"
ENCODER_FEATURE_CACHE = CACHE_ROOT / "encoder_features_v3"
CHECKPOINT_CACHE = CACHE_ROOT / "upstream_checkpoints/huggingface/pretrained"
RESULTS_DIR = RESULTS_ROOT / "encoder_probe_v3"

if not TOKEN_CACHE.exists():
    raise FileNotFoundError(f"V1 token cache not found: {TOKEN_CACHE}")
for path in (PACKED_TOKEN_CACHE, ENCODER_FEATURE_CACHE, CHECKPOINT_CACHE, RESULTS_DIR):
    path.mkdir(parents=True, exist_ok=True)
print("Token cache:", TOKEN_CACHE)
print("Packed token cache:", PACKED_TOKEN_CACHE)
print("V3 feature cache:", ENCODER_FEATURE_CACHE)
print("V3 results:", RESULTS_DIR)


In [ ]:
# 3) Pin the official source and cache its 5.1 MiB MTP-encoder checkpoint in Drive.
import hashlib, urllib.request

UPSTREAM_URL = "https://github.com/Jathurshan0330/TFM-Tokenizer.git"
UPSTREAM_REVISION = "2d6da482b16dabbb2ebec808fa9f505fc6f367c4"
UPSTREAM_ROOT = Path("/content/TFM-Tokenizer")
environment = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
if not UPSTREAM_ROOT.exists():
    run(["git", "clone", "--depth", "1", UPSTREAM_URL, str(UPSTREAM_ROOT)], env=environment)
run(["git", "fetch", "--depth", "1", "origin", UPSTREAM_REVISION], cwd=UPSTREAM_ROOT, env=environment)
run(["git", "checkout", "--detach", UPSTREAM_REVISION], cwd=UPSTREAM_ROOT, env=environment)
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_ROOT, text=True).strip()
if actual_revision != UPSTREAM_REVISION:
    raise RuntimeError(f"Official source revision mismatch: {actual_revision}")

ENCODER_CHECKPOINT = CHECKPOINT_CACHE / "tfm_encoder_mtp_last.pth"
ENCODER_SIZE = 5_308_635
ENCODER_SHA256 = "738d56a2021dd363c9d21c4804c441b486b28e33f73883dcf49623f9db8ad973"
ENCODER_URL = (
    "https://huggingface.co/Jathurshan/TFM-Tokenizer/resolve/"
    "e63f37850348b2c17429b5797442c0888163e4c9/"
    "pretrained/tfm_encoder_mtp_last.pth?download=true"
)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(2**20):
            digest.update(chunk)
    return digest.hexdigest()

checkpoint_valid = (
    ENCODER_CHECKPOINT.exists()
    and ENCODER_CHECKPOINT.stat().st_size == ENCODER_SIZE
    and file_sha256(ENCODER_CHECKPOINT) == ENCODER_SHA256
)
if not checkpoint_valid:
    temporary = ENCODER_CHECKPOINT.with_suffix(".download")
    print(f"Downloading official MTP encoder ({ENCODER_SIZE / 2**20:.1f} MiB)")
    urllib.request.urlretrieve(ENCODER_URL, temporary)
    if temporary.stat().st_size != ENCODER_SIZE or file_sha256(temporary) != ENCODER_SHA256:
        raise IOError("Downloaded MTP encoder failed size/SHA-256 verification")
    temporary.replace(ENCODER_CHECKPOINT)
else:
    print("Reusing verified Drive-cached MTP encoder")
print("Encoder checkpoint:", ENCODER_CHECKPOINT)
print("Official source revision:", actual_revision)


In [ ]:
# 4) Extract/resume frozen features, aggregate readers, and run the fixed V3 probe.
import importlib, json
import torch
import src.encoder_probe as encoder_probe_module
importlib.reload(encoder_probe_module)
from src.encoder_probe import (
    EncoderProbeConfig,
    OfficialFrozenTFMEncoder,
    build_sentence_features,
    evaluate_encoder_probe,
    extract_or_load_encoder_features,
)
from src.token_map import load_or_pack_token_records

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU, then rerun")

def save_runtime_stage(stage, **details):
    payload = {"stage": stage, **details}
    temporary = RESULTS_DIR / "runtime_status.tmp.json"
    temporary.write_text(json.dumps(payload, indent=2))
    temporary.replace(RESULTS_DIR / "runtime_status.json")
    print("Runtime stage:", stage, details)

config = EncoderProbeConfig()
save_runtime_stage("v3_started")
records, token_metadata, token_report = load_or_pack_token_records(
    TOKEN_CACHE, PACKED_TOKEN_CACHE, workers=8
)
save_runtime_stage("token_cache_loaded", recordings=len(records))
encoder = OfficialFrozenTFMEncoder(
    UPSTREAM_ROOT, ENCODER_CHECKPOINT, config=config, device="cuda"
)
encoder.report["official_source_revision"] = actual_revision
encoder.report["runtime_packages"] = {
    name: importlib.metadata.version(name)
    for name in ("torch", "einops", "linear-attention-transformer", "timm")
}
print("Encoder load report:", encoder.report)
save_runtime_stage("encoder_loaded", checkpoint_sha256=encoder.report["checkpoint_sha256"])
record_features, record_metadata, feature_report = extract_or_load_encoder_features(
    records,
    encoder,
    ENCODER_FEATURE_CACHE,
    dataset_fingerprint=token_report["dataset_fingerprint"],
    config=config,
)
record_metadata.to_csv(RESULTS_DIR / "record_feature_metadata.csv", index=False)
X, y, sentence_metadata = build_sentence_features(record_features, record_metadata)
sentence_metadata.to_csv(RESULTS_DIR / "sentence_metadata.csv", index=False)
print("Record features -> sentence features:", record_features.shape, "->", X.shape)
print("Feature report:", feature_report)
save_runtime_stage("features_complete", recordings=len(record_features), sentences=len(X))
metrics, predictions, summary, delta, gate = evaluate_encoder_probe(
    X=X,
    y=y,
    sentence_ids=sentence_metadata.sentence_id.to_numpy(),
    output_dir=RESULTS_DIR,
    cache_report=token_report,
    encoder_report=encoder.report,
    feature_report=feature_report,
    config=config,
)
display(summary)
print("Corrected paired bootstrap:", delta)
print("Decision:", gate["decision"])
save_runtime_stage("evaluation_complete", decision=gate["decision"])


In [ ]:
# 5) Read the saved result record and create the reproducible seed comparison.
import json
import matplotlib.pyplot as plt
import pandas as pd

metrics = pd.read_csv(RESULTS_DIR / "fold_metrics.csv")
summary = pd.read_csv(RESULTS_DIR / "summary.csv", header=[0, 1])
gate = json.loads((RESULTS_DIR / "viability_gate.json").read_text())
display(summary)
seed_scores = metrics.groupby(["seed", "setup"])["macro_f1"].mean().unstack("setup")
display(seed_scores)
axes = seed_scores[["encoder_probe", "encoder_probe_shuffled", "majority"]].plot.bar(figsize=(8, 4))
axes.axhline(1 / 3, color="black", linestyle="--", linewidth=1, label="1/3 reference")
axes.set(title="TFM V3 macro-F1 by split seed", ylabel="macro-F1", xlabel="seed")
axes.legend(loc="best")
plt.tight_layout()
figure_path = RESULTS_DIR / "macro_f1_by_seed.png"
plt.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print(json.dumps(gate, indent=2))
print("Saved figure:", figure_path)


## Interpretation boundary

V3 is still an unseen-sentence test for the known ZuCo reader pool, not an unseen-subject test. A failure means this fixed frozen TFM transfer sequence found no useful sentiment signal; it does not establish that EEG contains no sentiment information or that TFM fails on its original clinical benchmarks. V3 is the last planned version—there is no automatic V4 or post-hoc tuning loop.
